# Working with GerryChain

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

`PlanEvaluator` can score partitions after a run or expose the same metrics as GerryChain
updaters during the run. This example uses a short ReCom chain on a small grid so the two
workflows can be compared directly.

In [ ]:
from pathlib import Path

import networkx as nx
import pandas as pd
from gerrychain import MarkovChain, Partition
from gerrychain.proposals import ReCom
from gerrychain.updaters import Tally

import gerrytools.scoring as gs
from gerrytools.ben import RecordedChain, RecordedRun
from gerrytools.plotting import Histogram

## Build a small chain

The grid has 36 nodes and six starting districts. Population and two vote columns live on
the graph in the same form they would take on a state dual graph.

In [ ]:
grid = nx.grid_2d_graph(6, 6)
for row, column in grid:
    grid.nodes[(row, column)].update(
        population=1,
        district=(row * 6 + column) // 6,
        democratic=30 + 4 * column + row,
        republican=50 - 2 * column + row,
        x=column,
        y=row,
    )

Register the metrics once. `Tally` returns district populations, `Seats` returns one value
for the plan, and `CutEdges` returns a plan-level count. The explicit `cut_edge_count` name
keeps GerryTools' score distinct from GerryChain's built-in `cut_edges` updater.

In [ ]:
evaluator = gs.PlanEvaluator(grid).add_metrics(
    gs.Tally("population"),
    gs.Seats("democratic", "republican", result_name="democratic_seats"),
    gs.CutEdges(result_name="cut_edge_count"),
)

## Use the metrics as updaters

`to_updaters()` returns a mapping accepted directly by `Partition`. The first registered
metric requested on a partition computes all three scores; GerryChain then caches them on
that partition. Here the population result also supplies the updater required by ReCom.

In [ ]:
initial = Partition(grid, "district", updaters=evaluator.to_updaters())
chain = MarkovChain(
    ReCom.district_pairs_mst("population", pop_target=6, epsilon=0.15),
    initial_partition=initial,
    total_steps=4,
    rng=0,
)

The updater values are ordinary pandas objects or scalars. Access them by name while the
chain is running, alongside any other GerryChain updater.

In [ ]:
plans = list(chain)
pd.DataFrame(
    {
        "Democratic seats": [plan["democratic_seats"] for plan in plans],
        "Cut edges": [plan["cut_edge_count"] for plan in plans],
    },
)

## Evaluate selected plans together

The evaluator also accepts existing partitions. `evaluate_many()` is useful when only a few
plans need to be compared, whether they came from this chain, another run, or saved
assignments. `sample_ids` become the row labels in every returned table.

In [ ]:
selected = evaluator.evaluate_many(
    [plans[0], plans[-1]],
    sample_ids=["initial", "last"],
)
selected["population"]

In [ ]:
pd.DataFrame(
    {
        "Democratic seats": selected["democratic_seats"],
        "Cut edges": selected["cut_edge_count"],
    }
)

## Record, score, and compare complete chains

The same pieces form an end-to-end workflow for larger runs. Here two 1,000-step chains use
different ReCom variants on the same six-district grid. Each chain writes a standalone BENDL
recording under `stats/`. The overwrite calls make this fixed-path example safe to rerun.

In [ ]:
from tqdm.auto import tqdm

stats_dir = Path("stats")
stats_dir.mkdir(exist_ok=True)
recording_grid = nx.grid_2d_graph(10, 10)
recording_grid = nx.convert_node_labels_to_integers(recording_grid, ordering="sorted")
for node in recording_grid:
    recording_grid.nodes[node].update(population=1, district=node // 10)
proposals = {
    "district_pairs_ust": ReCom.district_pairs_ust("population", pop_target=10, epsilon=0),
    "cut_edges_mst": ReCom.cut_edges_mst("population", pop_target=10, epsilon=0),
}

recorded_chains = {}
for name, proposal in proposals.items():
    chain = RecordedChain(
        recording_grid,
        output_path=stats_dir / f"{name}.bendl",
        total_steps=10000,
        rng=0,
    )
    chain.initial_partition = Partition(
        chain.graph,
        "district",
        updaters={"population": Tally("population")},
    )
    chain.proposal_fn = proposal
    sum(1 for _ in tqdm(chain.allow_overwrite(), total=chain.total_steps))
    recorded_chains[name] = chain

{name: chain.recording.count_samples() for name, chain in recorded_chains.items()}

A stream evaluator reads each finalized recording and writes its scores to a separate run
directory. It does not reconstruct every `Partition` or hold the ensemble in memory.

In [ ]:
stream_evaluator = gs.PlanEvaluator(recorded_chains["district_pairs_ust"].graph).add_metric(
    gs.CutEdges()
)
score_paths = {}
for name, chain in recorded_chains.items():
    score_path = stats_dir / f"{name}_scores"
    stream_evaluator.evaluate_stream(chain.output_path, score_path, update=True)
    score_paths[name] = score_path

score_paths

The BENDL files and score directories are independent records. Reopen each BENDL file as a
`RecordedRun` when full partitions are needed, and reopen each score directory as an
`EnsembleEvalResult` when only saved measurements are needed.

In [ ]:
recorded_runs = {
    name: RecordedRun.from_bendl(chain.output_path) for name, chain in recorded_chains.items()
}
score_runs = {name: gs.EnsembleEvalResult.open(path) for name, path in score_paths.items()}
last_partitions = {
    name: run.partition_at(run.count_samples() - 1) for name, run in recorded_runs.items()
}
{name: len(partition.parts) for name, partition in last_partitions.items()}

Read the complete sample sequence with `expand_repetitions=True`. This restores repeated
Markov-chain steps before plotting the two cut-edge distributions.

In [ ]:
cut_edge_scores = pd.DataFrame(
    {
        name: run.read("cut_edges", expand_repetitions=True).reset_index(drop=True)
        for name, run in score_runs.items()
    }
)
plot = Histogram(
    legend=True,
    xlabel="Cut edges",
    ylabel="Plans",
    title="Recorded chain comparison",
)
plot.add_dataset(
    cut_edge_scores["district_pairs_ust"],
    name="District-pairs UST",
)
plot.add_dataset(
    cut_edge_scores["cut_edges_mst"],
    name="Cut-edges MST",
    facecolor="citizen_blue",
    facealpha=0.7,
)
plot.set_bin_widths(1)
plot.set_xlim(48, 82)
plot.center_bars()
plot.show()

The [BENDL scoring tutorial](bendl.ipynb) covers adding and replacing saved scores, result
shapes, batches, and array formulas. The [scoring overview](overview.ipynb) lists the
available metric families.